# 응용 모의고사 Set 3 — 정답 — 파생변수와 KNN 회귀

- 데이터: `mobiles.csv`
- 난이도: 기존 Set 01~06과 유사
- 구성: **공통 전처리 → Q1 통계 → Q2 상관분석 → Q3 모델링**
- 모든 문항은 공통 전처리 결과를 이어서 사용합니다.
- 전처리 완료 후 데이터는 **390행**이어야 합니다. 행 수가 다르면 다음 문제로 넘어가기 전에 전처리를 확인하세요.

정답 노트북은 `../answers/`에 있습니다.

## 공통 전처리 정답

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('../../dataset/mobiles.csv')
base = df.loc[df['num_rear_camera'] != 1].copy()
base['performance'] = (
    (base['RAM'] + base['ROM'])
    / (base['num_rear_camera'] + base['num_front_camera'])
)
assert len(base) == 390
display(base.head())

## Q1 정답

In [ ]:
cut = base['performance'].mean() + base['performance'].std()
answer_q1 = round(base.loc[base['performance'] > cut, 'discount_percent'].mean(), 3)
display(answer_q1)  # 0.098

## Q2 정답

In [ ]:
features = ['ratings', 'num_of_ratings', 'sales_price',
            'discount_percent', 'performance']
sales_corr = base[['sales'] + features].corr()['sales'].drop(index='sales')
answer_var_q2 = sales_corr.abs().idxmax()
answer_coef_q2 = round(sales_corr.loc[answer_var_q2], 2)
display(answer_var_q2, answer_coef_q2)  # num_of_ratings, 0.95

## Q3 정답

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import MinMaxScaler

features = ['ratings', 'num_of_ratings', 'sales_price',
            'discount_percent', 'performance', 'screen_size']
X = pd.get_dummies(base[features], columns=['screen_size'])
y = base['sales']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=321
)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
rmse_by_k = {}
for k in [2, 4, 6, 8, 10]:
    model = KNeighborsRegressor(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    rmse_by_k[k] = mean_squared_error(y_test, pred) ** 0.5
answer_q3 = pd.Series(rmse_by_k).idxmin()
display(pd.Series(rmse_by_k), answer_q3)  # 2